# SICD Orthorectification — Slant Range to Geographic Grid Projection

**Objective**: Project a SICD complex SAR image from slant-range geometry to a regular geographic grid (UTM or WGS-84) with optional DEM terrain correction.

## Overview

**Orthorectification** transforms imagery from sensor-native geometry (slant range, row/col pixels) to a **ground-referenced grid** (lat/lon or projected coordinates):
1. Define an **output grid** (UTM, WebMercator, or ENU)
2. For each output pixel, compute the corresponding **slant-range pixel** via geolocation inverse
3. **Resample** input data at those fractional pixel coordinates
4. Apply **DEM terrain correction** (optional) to map pixels to true ground height

## Theory

### Slant Range vs Ground Range

SAR imagery is natively in **slant range** — the line-of-sight distance from sensor to ground:
$$R_{slant} = \sqrt{R_{ground}^2 + h^2}$$
where $h$ is platform altitude.

Orthorectification reverses this: given a ground point $(lat, lon, h_{DEM})$, solve for the slant-range pixel $(row, col)$.

### DEM Terrain Correction

Without DEM:
- Ground points assumed at **ellipsoid height** (h = 0)
- Produces **layover** and **foreshortening** artifacts in mountainous terrain

With DEM:
- Each ground point uses **DEM-queried elevation** $h_{DEM}(lat, lon)$
- SICD R/Rdot inverse iteratively refines position using $h_{DEM}$
- Corrects geometric distortions from terrain relief

### Output Grids

GRDL supports three output projections:

| Grid Type | Coordinate System | Use Case |
|-----------|------------------|----------|
| **UTMGrid** | UTM zone (meters) | Local-scale analysis, preserves distances |
| **WebMercatorGrid** | EPSG:3857 (web maps) | Overlay on slippy maps (Google/OSM) |
| **ENUGrid** | East-North-Up (meters) | Radar-centric, local Cartesian |

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | SICD reader via factory (`get_reader('sicd', ...)`) |
| `grdl.geolocation` | `create_geolocation()` auto-detect, DEM attachment |
| `grdl.geolocation.elevation` | `open_elevation()` for DEM loading |
| `grdl.image_processing.ortho` | `orthorectify()`, `UTMGrid`, `ENUGrid` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Imports

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid, ENUGrid
from grdl.contrast import PercentileStretch

## Configuration — User Paths

In [ ]:
# Path to SICD file
SICD_PATH = Path("/path/to/your/sicd_file.nitf")

# Optional: DEM path for terrain correction
# Set to None to use ellipsoid (h=0)
DEM_PATH = None  # e.g., Path("/path/to/fabdem/tiles") or Path("/path/to/srtm.tif")

# Validate path
assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"

print(f"✓ SICD file: {SICD_PATH.name}")
if DEM_PATH:
    print(f"✓ DEM: {DEM_PATH}")
else:
    print("⚠ No DEM — using ellipsoid height (h=0)")

## Step 1: Load SICD Metadata and Extract Chip

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = meta.image_data.num_rows, meta.image_data.num_cols
    
    print(f"Image size: {rows} × {cols}")
    print(f"Pixel type: {meta.image_data.pixel_type}")
    print(f"Collection mode: {meta.collection_info.radar_mode.mode_type}")
    
    # Extract center chip (4096×4096 or smaller)
    chip_size = min(4096, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    chip = reader.read_chip(
        row_start=r_start, row_end=r_start + chip_size,
        col_start=c_start, col_end=c_start + chip_size,
    )

print(f"\nExtracted chip: {chip.shape}, dtype: {chip.dtype}")
print(f"Chip location: rows [{r_start}, {r_start + chip_size}), cols [{c_start}, {c_start + chip_size})")

## Step 2: Create Geolocation Object

**Critical**: For chips, wrap the geolocation with `ChipGeolocation` to handle row/col offset.

In [ ]:
from grdl.geolocation.chip import ChipGeolocation

with get_reader('sicd', SICD_PATH) as reader:
    # Create geolocation for full image
    geo_full = create_geolocation(reader)
    
    # Wrap with chip offset
    geo = ChipGeolocation(
        geolocation=geo_full,
        row_offset=r_start,
        col_offset=c_start,
        shape=(chip_size, chip_size),
    )

print(f"✓ Geolocation: {type(geo_full).__name__}")
print(f"  Chip offset: row={r_start}, col={c_start}")

## Step 3 (Optional): Attach DEM to Geolocation

**Per GRDL architecture**: The geolocation object owns the DEM, not the orthorectifier.

In [ ]:
if DEM_PATH:
    from grdl.geolocation.elevation import open_elevation
    
    # Attach DEM to the wrapped geolocation's inner geolocation
    geo.geolocation.elevation = open_elevation(DEM_PATH)
    print(f"✓ DEM attached: {type(geo.geolocation.elevation).__name__}")
else:
    print("⚠ No DEM attached — projection uses ellipsoid (h=0)")

## Step 4: Compute Magnitude in Slant Range

**Important**: Compute magnitude *before* resampling. Resampling complex data causes phase cancellation.

In [ ]:
magnitude = np.abs(chip)

# Convert to dB (logarithmic compression for SAR backscatter)

magnitude_db = 20 * np.log10(magnitude + 1e-12)
print(f"Magnitude (dB) range: [{magnitude_db.min():.1f}, {magnitude_db.max():.1f}]")


## Step 5: Define Output Grid — UTM Projection

In [ ]:
# UTM grid from geolocation footprint
output_grid = UTMGrid.from_geolocation(
    geolocation=geo,
    pixel_size_m=0.5,           # 0.5 m ground sample distance
    margin_m=0.0,               # No extra margin
)

print(f"UTM Grid:")
print(f"  Zone: {output_grid.zone} {'N' if output_grid.north else 'S'}")
print(f"  Size: {output_grid.rows} × {output_grid.cols}")
print(f"  Pixel size: {output_grid.pixel_size} m")

## Step 6: Orthorectify to UTM Grid

In [ ]:
print("Running orthorectification...")

ortho_result = orthorectify(
    geolocation=geo,
    source_array=magnitude_db,
    output_grid=output_grid,
    interpolation='bilinear',  # or 'nearest', 'bicubic'
)

ortho_db = ortho_result.data

print(f"✓ Orthorectified: {ortho_db.shape}")
print(f"  Valid pixels: {np.sum(~np.isnan(ortho_db))} / {ortho_db.size}")
print(f"  NaN pixels (outside footprint): {np.sum(np.isnan(ortho_db))}")

## Step 7: Visualization — Slant Range vs UTM Ortho Comparison

In [ ]:
# Apply percentile stretch for display (clips outliers, auto-contrast)
stretch = PercentileStretch(plow=2.0, phigh=98.0)
slant_display = stretch.apply(magnitude_db)
ortho_display = stretch.apply(ortho_db)

viewer = napari.Viewer(title="SICD Orthorectification")

# Layer 1: Original slant-range (rectangular chip, full coverage)
slant_layer = viewer.add_image(
    slant_display,
    name="Slant Range (rectangular)",
    colormap="gray",
    visible=False,
)

# Layer 2: Orthorectified UTM grid (diamond-shaped valid data)
ortho_layer = viewer.add_image(
    ortho_display,
    name="UTM Ortho (diamond)",
    colormap="gray",
    visible=True,
)

# Link layers for mutually exclusive visibility
def toggle_layers(layer):
    """When one layer becomes visible, hide the other."""
    if layer.visible:
        if layer == slant_layer:
            ortho_layer.visible = False
        else:
            slant_layer.visible = False

slant_layer.events.visible.connect(lambda event: toggle_layers(slant_layer))
ortho_layer.events.visible.connect(lambda event: toggle_layers(ortho_layer))

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer opened")
print("  Layer 1: Slant Range — rectangular chip (original sensor geometry, full coverage)")
print("  Layer 2: UTM Ortho — diamond shape (georeferenced grid, NaN outside chip boundary)")
print("\n💡 Toggle between layers using the eye icons — only one visible at a time.")
print("\n⚠ Black regions in UTM layer are NaN (chip boundaries don't align with UTM axes).")
print("  Slant range is a rectangular chip; ortho is diamond-shaped after projection.")
print("\nExecution paused — close the napari window to continue and free memory...")
napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del chip, magnitude, magnitude_db, slant_display, ortho_db, ortho_display, geo, geo_full, stretch, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Physical Interpretation

### Layer Comparison

Toggle between layers to see the transformation:

**Slant Range (Layer 1)**:
- **Rectangular chip** — the extracted 4096×4096 pixel subset
- Sensor-native geometry (row/col coordinates)
- Full coverage, no NaN pixels — every pixel has data
- **Foreshortening**: Slopes facing radar appear compressed
- **Layover**: Tall objects lean toward radar
- **Shadow**: Areas behind tall objects are dark

**UTM Ortho (Layer 2)**:
- **Diamond-shaped valid data region** with black NaN outside
- Same data as slant range, reprojected to UTM grid
- Black pixels are NaN where the rectangular chip boundary doesn't align with UTM axes
- Georeferenced to true ground coordinates (easting/northing in meters)
- With DEM: terrain relief corrected
- Without DEM: ellipsoid projection (h=0)

### Why Diamond-Shaped Ortho from Rectangular Chip?

The **chip is rectangular in slant range** (row/col coordinates), but when projected to **UTM** (easting/northing):
- Chip corners map to non-aligned UTM positions
- Result: rotated parallelogram/diamond shape in UTM grid
- Areas outside this diamond are NaN (no source data at those UTM coordinates)

### Orthorectified Correction

**With DEM** (if attached):
- Terrain relief corrected — features align with true ground coordinates
- Layover and foreshortening geometrically compensated

**Without DEM** (ellipsoid projection, as in this notebook):
- Ground points assumed at h=0 (WGS-84 ellipsoid)
- Layover/shadow artifacts remain but positions are geocoded

### Invalid Pixels (NaN)

NaN pixels in ortho output occur when:

- Ground point is **outside image footprint** (edge of output grid)- **ENU**: Radar-centric Cartesian (East-North-Up from SCP) — ideal for local analysis

- DEM has **no coverage** at that location- **WebMercator**: Compatible with web maps (Google/OSM) but distorts at high latitudes

- Geometry is **unsolvable** (e.g., layover region with multiple solutions)- **UTM**: Preserves distances and angles within a zone (±500 km from central meridian)


### UTM vs WebMercator vs ENU

## Summary

Successfully demonstrated:
- ✅ SICD slant-range image loading via IO factory
- ✅ Chip extraction with offset-aware geolocation (`ChipGeolocation`)
- ✅ DEM attachment to geolocation object (architecture compliance)
- ✅ UTM grid definition and orthorectification
- ✅ Magnitude computation before resampling (avoids phase cancellation)
- ✅ dB + percentile stretch for SAR display (best practice)
- ✅ Mutually exclusive layer comparison (slant rectangular vs ortho diamond)

### Key GRDL Patterns
- `get_reader('sicd', path)` for format-agnostic loading
- `create_geolocation(reader)` for auto-detection
- **DEM ownership**: `geo.elevation = open_elevation(dem_path)`
- `ChipGeolocation` wrapper for chip offsets
- `orthorectify(data, geolocation, output_grid)` for single-call projection

### Next Steps
- Try `ENUGrid` for radar-centric local projection
- Attach DEM for terrain-corrected output
- Increase pixel_size_m (1.0, 2.0) for faster processing
- Export to GeoTIFF: `GeoTIFFWriter.write(ortho_result.data, ...)`
- Compare with SIDD ortho (pre-projected product, next notebook)